# 第 16 章 確率的ソロー・モデル: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/macro-programming/16-stochastic-solow.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
alpha = 0.36         # 資本分配率 α
saving = 0.2         # 貯蓄率 s
depreciation = 0.025  # 減耗率 δ
rho = 0.6            # TFP のショックの持続性 ρ
sigma = 0.008        # TFP のショックの標準偏差 σ
hours = 1.0          # 総労働時間 H
n = 200              # シミュレーションの期間
seed = 123           # 乱数の種

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

## シミュレーション

In [ ]:
import numpy as np

from lib.chapter16.solow_stochastic import (
    SolowParameters,
    log_deviations,
    stochastic_solow,
)

params = SolowParameters(
    alpha=alpha,
    saving=saving,
    depreciation=depreciation,
    rho=rho,
    sigma=sigma,
    hours=hours,
)
dev = log_deviations(stochastic_solow(params, n, np.random.default_rng(seed)), params)
print("GDP の乖離の標準偏差（%）:", round(float(dev["y"].std()), 4))
print("消費 ÷ GDP の標準偏差の比:", round(float(dev["c"].std() / dev["y"].std()), 4))
print("投資 ÷ GDP の標準偏差の比:", round(float(dev["i"].std() / dev["y"].std()), 4))

## 図

In [ ]:
from lib.chapter16.plot import plot_solow_simulation

plot_solow_simulation(dev)

## 景気循環の分解（データ）

In [ ]:
from lib.chapter16.plot import plot_decomposition
from lib.chapter16.solow_stochastic import (
    business_cycle_components,
    decompose_business_cycle,
    load_solow_quarterly,
)

cycles = business_cycle_components(load_solow_quarterly(), alpha)
shares = decompose_business_cycle(cycles, alpha)
print("GDP の分散に占める割合:", shares.round(3).to_dict())
plot_decomposition(shares)

## 試してみよう

- `rho = 0.9` にすると、
  GDP の乖離の標準偏差は 0.885 から 1.302 に増えます（ショックが長く残ります）
- `sigma = 0.016` にすると、標準偏差は 1.77 で、ほぼ 2 倍です
- `seed = 1` にすると、標準偏差は 0.8054 です。消費・投資の比は、
  種や貯蓄率を変えても、1.0 のままです（貯蓄率が一定だからです）
- `alpha = 0.5` にすると、データの分解は、TFP が 0.676、労働が 0.328 に変わります